In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [ ]:
# from langchain_mcp_adapters.client import MultiServerMCPClient

# client = MultiServerMCPClient(
#     {
#         "local_server": {
#                 "transport": "stdio",
#                 "command": "python",
#                 "args": ["resources/2.1_mcp_server.py"],
#             }
#     }
# )

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com",
            }
    }
)

In [10]:
from pprint import pprint

# get tools
tools = await client.get_tools()

pprint(tools)

[StructuredTool(name='search-flight', description='\n# Search for a flight\n\n## Description\n\nUses the Kiwi API to search for available flights between two locations on a specific date.\n\n## How it works\n\nThe tool will:\n1. Search for matching locations to resolve airport codes\n2. Find available flights for the specified route and date range\n\n## Method\n\nCall this tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just city names.\n\nYou should display the returned results in a markdown table format: Group the results by price (those who are the cheapest), duration (those who are the shortest, i.e. have the smallest \'totalDurationInSeconds\') and the rest (those that could still be interesting).\n\nAlways display for each flight in order:\n  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")\n  - In the 2nd column: The departure and arrival dates 

In [6]:
# # get tools
# tools = await client.get_tools()

# # get resources
# resources = await client.get_resources("local_server")

# # get prompts
# prompt = await client.get_prompt("local_server", "prompt")
# prompt = prompt[0].content

  + Exception Group Traceback (most recent call last):
  |   File "/Users/hsouza/play/htssouza/ai-langchain-agents/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3745, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/bf/nm95t6rn2z15zsjw6qfd60d80000gp/T/ipykernel_61965/3663964148.py", line 5, in <module>
  |     resources = await client.get_resources("local_server")
  |                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/hsouza/play/htssouza/ai-langchain-agents/.venv/lib/python3.13/site-packages/langchain_mcp_adapters/client.py", line 238, in get_resources
  |     async with self.session(server_name) as session:
  |                ~~~~~~~~~~~~^^^^^^^^^^^^^
  |   File "/opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/contextlib.py", line 235, in __aexit__
  |     await self.gen.athrow(value)
  |   File "/Users/hsouza/play/htssouza/ai-la

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="You are a helpful assistant that can use tools to answer questions about flights."
)

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about flights between Manchester and Lisbon?")]},
    config=config
)

In [13]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about flights between Manchester and Lisbon?', additional_kwargs={}, response_metadata={}, id='92972f94-2d4f-4e56-b9cb-b058029dc4b1'),
              AIMessage(content='I can help with that. To fetch live flight options, I just need a few details:\n\n- Departure date (dd/mm/yyyy). If you’re flexible, tell me a date range (e.g., +/- 3 days).\n- Return date (if round-trip) in the same format, or indicate one-way.\n- Number of passengers (adults, children, infants).\n- Cabin class (Economy, Premium Economy, Business, First).\n- Preference for direct flights only, or OK with one or more layovers.\n- Currency for prices (EUR is fine default) and any language preference for results.\n\nIf you’d like, I can start with the next 7 days as a quick sample search. Just say “start a sample search” and I’ll pull the latest options.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 873, 'prompt_tokens': 1227, 'tot

## Online MCP

In [14]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [15]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [16]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='2c15fa28-6b65-4696-834b-8c4d9475323f'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 296, 'total_tokens': 451, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DSNuiYIygKgLp0h8FiQZDx88VgqIT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6d6e-e813-7ec0-b2ab-290da55da867-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_jwpV3zjCK7QJM7i2gwk4Cd8B', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':